In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from PROPS_EV.calculateEVS import *
from BACKTEST.backtest import *
from MODELS.pipeline import *

### Load Model

In [2]:
PTSmodel = joblib.load('Models/xgbPTSModel.pkl')
PTSfeatures = joblib.load('Models/topPTSfeatures.pkl')
REBmodel = joblib.load('Models/xgbREBModel.pkl')
REBfeatures = joblib.load('Models/topREBfeatures.pkl')

### Load Data

In [12]:
pd.set_option('display.max_columns', None)


s25_pts = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv')
s24_pts = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_24.csv')
s25_reb = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/REB_TRAIN_25.csv')
s24_reb = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/REB_TRAIN_24.csv')

dfPTS = pd.concat([s25_pts, s24_pts]).sort_values(by='GAME_DATE')
dfREB = pd.concat([s25_reb, s24_reb]).sort_values(by='GAME_DATE')


# date = '2025-02-12'
# dfPTS = dfPTS[dfPTS['GAME_DATE'] < date]
# dfREB = dfREB[dfREB['GAME_DATE'] < date]

dfsData = pd.read_csv('../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_20251016.csv')
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]
dfsREB = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_rebounds')]
dfsPTS.rename(columns={'OVER/UNDER':'SIDE'}, inplace=True)
dfsPTS

C:\Users\alexg\AppData\Local\Temp\ipykernel_22452\2170523533.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfsPTS.rename(columns={'OVER/UNDER':'SIDE'}, inplace=True)


,BOOKMAKER,CATEGORY,NAME,SIDE,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
79,Underdog,player_points,Kevin Durant,Over,25.5,-137,2025-10-21T23:35:00Z,2025-10-17T01:00:59Z
80,Underdog,player_points,Kevin Durant,Under,25.5,-137,2025-10-21T23:35:00Z,2025-10-17T01:00:59Z
81,Underdog,player_points,Shai Gilgeous-Alexander,Over,32.5,-137,2025-10-21T23:35:00Z,2025-10-17T01:00:59Z
82,Underdog,player_points,Shai Gilgeous-Alexander,Under,32.5,-137,2025-10-21T23:35:00Z,2025-10-17T01:00:59Z
83,Underdog,player_points,Alperen Sengun,Over,18.5,-137,2025-10-21T23:35:00Z,2025-10-17T01:00:59Z
...,...,...,...,...,...,...,...,...
242,Underdog,player_points,Julian Champagnie,Under,10.5,-137,2025-10-23T01:40:00Z,2025-10-17T01:01:18Z
243,Underdog,player_points,Devin Booker,Over,28.5,-137,2025-10-23T02:10:00Z,2025-10-17T01:02:03Z
244,Underdog,player_points,Devin Booker,Under,28.5,-137,2025-10-23T02:10:00Z,2025-10-17T01:02:03Z
245,Underdog,player_points,Dillon Brooks,Over,14.5,-137,2025-10-23T02:10:00Z,2025-10-17T01:02:03Z


In [5]:
backtestData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/singleBookies.csv')
backtestData = backtestData[(backtestData['ODDS'] <= 200) & (backtestData['ODDS'] >= -200)]
singlePTSBookies = backtestData[(backtestData['CATEGORY'] == 'points') & (backtestData['GAME_DATE'] == date)]
singleREBBookies = backtestData[(backtestData['CATEGORY'] == 'rebounds') & (backtestData['GAME_DATE'] == date)]

### Top EVs for single bets

In [6]:
results = single_bet(
    data=dfPTS,
    bookmakers=singlePTSBookies,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=4.5,
    stake=5,
    simulations=10000, 
    std_window=10,
    min_std=2.0,
    max_std=10.0,
    stat_col='PTS'
)
results.sort_values(by='EV%', ascending=False).head(10)

Processing single bets...


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,RECOMMENDATION,OVER%,UNDER%,IMPLIED PROB,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER,CONFIDENCE INTERVAL
1621,Toumani Camara,betmgm,points,5.5,125,over,15.211010,1,0.997,0.003,0.444,124.30,0.99,0.50,0.25,"(8.6, 22.0)"
1540,Steven Adams,betmgm,points,2.5,125,over,5.820261,0,0.949,0.051,0.444,113.50,0.91,0.45,0.23,"(1.8, 9.7)"
1417,Jalen Green,draftkings,points,22.5,135,under,14.470527,1,0.128,0.872,0.426,105.01,0.78,0.39,0.19,"(2.6, 28.1)"
382,DaQuan Jeffries,betmgm,points,2.5,110,over,6.335608,0,0.973,0.027,0.476,104.37,0.95,0.47,0.24,"(2.5, 10.3)"
1541,Steven Adams,espnbet,points,2.5,110,over,5.820261,0,0.950,0.050,0.476,99.52,0.90,0.45,0.23,"(1.9, 9.8)"
1679,Zeke Nnaji,draftkings,points,12.5,100,under,5.575228,1,0.016,0.984,0.500,96.84,0.97,0.48,0.24,"(0.7, 11.9)"
1426,Alperen Sengun,espnbet,points,19.5,-110,under,7.601803,1,0.001,0.999,0.524,90.76,1.00,0.50,0.25,"(1.2, 15.1)"
1389,Jaylin Williams,draftkings,points,5.5,-105,over,10.058649,1,0.974,0.026,0.512,90.24,0.95,0.47,0.24,"(5.5, 14.6)"
1390,Jaylin Williams,betmgm,points,4.5,-110,over,10.058649,1,0.993,0.007,0.524,89.48,0.98,0.49,0.25,"(5.6, 14.5)"
441,Bilal Coulibaly,betmgm,points,7.5,105,over,14.761505,1,0.912,0.088,0.488,86.90,0.83,0.41,0.21,"(4.6, 25.4)"


In [7]:
results = single_bet(
    data=dfREB,
    bookmakers=singleREBBookies,
    model=REBmodel,
    features=REBfeatures,
    edge_threshold=2.0,
    stake=5,
    simulations=10000, 
    std_window=10,
    min_std=1.5,
    max_std=6.5,
    stat_col='REB'
)
results.sort_values(by='EV%', ascending=False).head(10)

Processing single bets...


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,RECOMMENDATION,OVER%,UNDER%,IMPLIED PROB,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER,CONFIDENCE INTERVAL
412,Thomas Bryant,betmgm,rebounds,10.5,125,under,5.142313,1,0.006,0.994,0.444,123.74,0.99,0.49,0.25,"(1.4, 9.2)"
1531,Kris Murray,betmgm,rebounds,4.5,135,under,1.732772,1,0.101,0.899,0.426,111.24,0.82,0.41,0.21,"(0.1, 6.0)"
1448,Donovan Clingan,espnbet,rebounds,9.5,105,under,3.203625,1,0.002,0.998,0.488,104.61,1.00,0.50,0.25,"(0.3, 7.4)"
1555,Zeke Nnaji,espnbet,rebounds,5.5,115,under,2.452899,1,0.069,0.931,0.465,100.12,0.87,0.44,0.22,"(0.2, 6.4)"
1726,Olivier-Maxence Prosper,fanduel,rebounds,8.5,100,under,1.889173,1,0.001,0.999,0.500,99.88,1.00,0.50,0.25,"(0.2, 6.0)"
1557,Zeke Nnaji,fanduel,rebounds,5.5,112,under,2.452899,1,0.073,0.927,0.472,96.59,0.86,0.43,0.22,"(0.2, 6.5)"
1450,Donovan Clingan,betmgm,rebounds,9.5,-105,under,3.203625,1,0.002,0.998,0.512,94.89,1.00,0.50,0.25,"(0.3, 7.5)"
229,Keldon Johnson,draftkings,rebounds,2.5,150,over,4.210320,0,0.777,0.223,0.400,94.25,0.63,0.31,0.16,"(0.5, 9.7)"
1156,Bobby Portis,betmgm,rebounds,6.5,115,over,9.395657,1,0.903,0.097,0.465,94.10,0.82,0.41,0.20,"(5.0, 13.7)"
409,Thomas Bryant,espnbet,rebounds,8.5,105,under,5.142313,1,0.053,0.947,0.488,94.07,0.90,0.45,0.22,"(1.3, 9.2)"


### Top EVs for 2 leg bets

In [13]:
results = prizepickspairsEV(
    data=dfPTS,
    bookmakers=dfsPTS,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=5,
    stake=100,
    simulations=1000,
    std_window=10,
    min_std=2.0,
    max_std=10.0,
    stat_col='PTS'
)
results.sort_values(by='EV%', ascending=False).head(10).reset_index(drop=True)

Processing pairs...


,PLAYER 1,CATEGORY 1,BOOKMAKER 1,ODDS 1,LINE 1,PREDICTION 1,MODEL_SIDE 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,BOOKMAKER 2,ODDS 2,LINE 2,PREDICTION 2,MODEL_SIDE 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
0,Mikal Bridges,player_points,Underdog,-137,15.5,4.02,UNDER,0.035,0.965,"(0.3, 16.1)",Trey Murphy III,player_points,Underdog,-137,19.5,1.55,UNDER,0.006,0.994,"(0.2, 16.7)",UNDER/UNDER,1,0.9592,1.878,0.939
1,James Harden,player_points,Underdog,-137,20.5,28.37,OVER,0.934,0.066,"(18.2, 38.7)",Trey Murphy III,player_points,Underdog,-137,19.5,1.55,UNDER,0.006,0.994,"(0.2, 16.7)",OVER/UNDER,1,0.9284,1.785,0.893
2,Lauri Markkanen,player_points,Underdog,-137,21.5,14.34,UNDER,0.076,0.924,"(5.1, 23.8)",Trey Murphy III,player_points,Underdog,-137,19.5,1.55,UNDER,0.006,0.994,"(0.2, 16.7)",UNDER/UNDER,1,0.9185,1.755,0.878
3,Jarrett Allen,player_points,Underdog,-137,12.5,3.31,UNDER,0.076,0.924,"(0.5, 14.7)",Trey Murphy III,player_points,Underdog,-137,19.5,1.55,UNDER,0.006,0.994,"(0.2, 16.7)",UNDER/UNDER,1,0.9185,1.755,0.878
4,Bobby Portis,player_points,Underdog,-137,10.5,15.92,OVER,0.910,0.090,"(7.3, 24.7)",Trey Murphy III,player_points,Underdog,-137,19.5,1.55,UNDER,0.006,0.994,"(0.2, 16.7)",OVER/UNDER,1,0.9045,1.714,0.857
5,James Harden,player_points,Underdog,-137,20.5,28.37,OVER,0.934,0.066,"(18.2, 38.7)",Mikal Bridges,player_points,Underdog,-137,15.5,4.02,UNDER,0.035,0.965,"(0.3, 16.1)",OVER/UNDER,1,0.9013,1.704,0.852
6,Julian Champagnie,player_points,Underdog,-137,10.5,16.91,OVER,0.905,0.095,"(8.0, 25.9)",Trey Murphy III,player_points,Underdog,-137,19.5,1.55,UNDER,0.006,0.994,"(0.2, 16.7)",OVER/UNDER,1,0.8996,1.699,0.849
7,Quentin Grimes,player_points,Underdog,-137,13.5,19.09,OVER,0.901,0.099,"(10.7, 27.5)",Trey Murphy III,player_points,Underdog,-137,19.5,1.55,UNDER,0.006,0.994,"(0.2, 16.7)",OVER/UNDER,1,0.8956,1.687,0.843
8,Jarrett Allen,player_points,Underdog,-137,12.5,3.31,UNDER,0.076,0.924,"(0.5, 14.7)",Mikal Bridges,player_points,Underdog,-137,15.5,4.02,UNDER,0.035,0.965,"(0.3, 16.1)",UNDER/UNDER,1,0.8917,1.675,0.837
9,Lauri Markkanen,player_points,Underdog,-137,21.5,14.34,UNDER,0.076,0.924,"(5.1, 23.8)",Mikal Bridges,player_points,Underdog,-137,15.5,4.02,UNDER,0.035,0.965,"(0.3, 16.1)",UNDER/UNDER,1,0.8917,1.675,0.837


In [10]:
results = prizepickspairsEV(
    data=dfREB,
    bookmakers=dfsREB,
    model=REBmodel,
    features=REBfeatures,
    edge_threshold=2.0,
    stake=100,
    simulations=10000,
    std_window=10,
    min_std=1.5,
    max_std=6.5,
    stat_col='REB'
)
results.sort_values(by='EV%', ascending=False).head(10).reset_index(drop=True)

Processing pairs...


,PLAYER 1,CATEGORY 1,BOOKMAKER 1,ODDS 1,LINE 1,PREDICTION 1,MODEL_SIDE 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,BOOKMAKER 2,ODDS 2,LINE 2,PREDICTION 2,MODEL_SIDE 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
0,Precious Achiuwa,player_rebounds,prizepicks,-137,8.5,7.05,UNDER,0.226,0.774,"(3.1, 10.9)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.802,0.198,"(3.1, 16.7)",UNDER/OVER,0,0.6214,0.864,0.432
1,Josh Hart,player_rebounds,prizepicks,-137,9.5,7.59,UNDER,0.238,0.762,"(2.4, 12.9)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.802,0.198,"(3.1, 16.7)",UNDER/OVER,0,0.6118,0.835,0.418
2,Pascal Siakam,player_rebounds,prizepicks,-137,7.5,5.85,UNDER,0.259,0.741,"(1.2, 11.0)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.802,0.198,"(3.1, 16.7)",UNDER/OVER,0,0.5949,0.785,0.392
3,Thomas Bryant,player_rebounds,prizepicks,-137,6.5,5.14,UNDER,0.263,0.737,"(1.3, 9.2)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.802,0.198,"(3.1, 16.7)",UNDER/OVER,0,0.5914,0.774,0.387
4,Immanuel Quickley,player_rebounds,prizepicks,-137,3.5,4.89,OVER,0.730,0.270,"(0.8, 9.5)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.802,0.198,"(3.1, 16.7)",OVER/OVER,0,0.5857,0.757,0.378
5,Jalen Duren,player_rebounds,prizepicks,-137,11.5,14.22,OVER,0.727,0.273,"(5.5, 22.8)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.802,0.198,"(3.1, 16.7)",OVER/OVER,1,0.5836,0.751,0.375
6,Pascal Siakam,player_rebounds,prizepicks,-137,7.5,5.85,UNDER,0.259,0.741,"(1.2, 11.0)",Precious Achiuwa,player_rebounds,prizepicks,-137,8.5,7.05,UNDER,0.226,0.774,"(3.1, 10.9)",UNDER/UNDER,0,0.5740,0.722,0.361
7,Precious Achiuwa,player_rebounds,prizepicks,-137,8.5,7.05,UNDER,0.226,0.774,"(3.1, 10.9)",Thomas Bryant,player_rebounds,prizepicks,-137,6.5,5.14,UNDER,0.263,0.737,"(1.3, 9.2)",UNDER/UNDER,0,0.5706,0.712,0.356
8,Precious Achiuwa,player_rebounds,prizepicks,-137,8.5,7.05,UNDER,0.226,0.774,"(3.1, 10.9)",Santi Aldama,player_rebounds,prizepicks,-137,5.5,7.50,OVER,0.733,0.267,"(1.6, 14.1)",UNDER/OVER,0,0.5677,0.703,0.352
9,Josh Hart,player_rebounds,prizepicks,-137,9.5,7.59,UNDER,0.238,0.762,"(2.4, 12.9)",Pascal Siakam,player_rebounds,prizepicks,-137,7.5,5.85,UNDER,0.259,0.741,"(1.2, 11.0)",UNDER/UNDER,0,0.5652,0.696,0.348


## 3 leg parlay

In [14]:
threeLeg = prizepicks3LegEV(
    data=dfPTS,
    bookmakers=dfsPTS,
    model=PTSmodel,
    features=PTSfeatures,
    edge_threshold=4.5,
    stake=100,
    simulations=10000,
    std_window=10,
    min_std=2.0,
    max_std=10.0,
    stat_col='PTS'
)
threeLeg.sort_values(by='EV%', ascending=False).head(10)

Processing 3-leg parlays...


,PLAYER 1,CATEGORY 1,BOOKMAKER 1,ODDS 1,LINE 1,PREDICTION 1,MODEL_SIDE 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,BOOKMAKER 2,ODDS 2,LINE 2,PREDICTION 2,MODEL_SIDE 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,PLAYER 3,CATEGORY 3,BOOKMAKER 3,ODDS 3,LINE 3,PREDICTION 3,MODEL_SIDE 3,OVER% 3,UNDER% 3,CONFIDENCE INTERVAL 3,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
48555,Jarrett Allen,player_points,Underdog,-137,12.5,3.31,UNDER,0.069,0.931,"(0.3, 14.9)",Mikal Bridges,player_points,Underdog,-137,15.5,4.02,UNDER,0.044,0.956,"(0.4, 17.2)",Trey Murphy III,player_points,Underdog,-137,19.5,1.55,UNDER,0.005,0.995,"(0.3, 16.2)",UNDER/UNDER/UNDER,1,0.8855,4.313,0.863
47994,James Harden,player_points,Underdog,-137,20.5,28.37,OVER,0.927,0.073,"(18.0, 38.7)",Mikal Bridges,player_points,Underdog,-137,15.5,4.02,UNDER,0.044,0.956,"(0.4, 17.2)",Trey Murphy III,player_points,Underdog,-137,19.5,1.55,UNDER,0.005,0.995,"(0.3, 16.2)",OVER/UNDER/UNDER,1,0.8812,4.287,0.857
53206,Lauri Markkanen,player_points,Underdog,-137,21.5,14.34,UNDER,0.081,0.919,"(4.3, 24.4)",Mikal Bridges,player_points,Underdog,-137,15.5,4.02,UNDER,0.044,0.956,"(0.4, 17.2)",Trey Murphy III,player_points,Underdog,-137,19.5,1.55,UNDER,0.005,0.995,"(0.3, 16.2)",UNDER/UNDER/UNDER,1,0.8735,4.241,0.848
51261,Julian Champagnie,player_points,Underdog,-137,10.5,16.91,OVER,0.907,0.093,"(7.5, 26.4)",Mikal Bridges,player_points,Underdog,-137,15.5,4.02,UNDER,0.044,0.956,"(0.4, 17.2)",Trey Murphy III,player_points,Underdog,-137,19.5,1.55,UNDER,0.005,0.995,"(0.3, 16.2)",OVER/UNDER/UNDER,1,0.8625,4.175,0.835
53795,Mikal Bridges,player_points,Underdog,-137,15.5,4.02,UNDER,0.044,0.956,"(0.4, 17.2)",Quentin Grimes,player_points,Underdog,-137,13.5,19.09,OVER,0.904,0.096,"(10.8, 27.4)",Trey Murphy III,player_points,Underdog,-137,19.5,1.55,UNDER,0.005,0.995,"(0.3, 16.2)",UNDER/OVER/UNDER,1,0.8601,4.161,0.832
47587,James Harden,player_points,Underdog,-137,20.5,28.37,OVER,0.927,0.073,"(18.0, 38.7)",Jarrett Allen,player_points,Underdog,-137,12.5,3.31,UNDER,0.069,0.931,"(0.3, 14.9)",Trey Murphy III,player_points,Underdog,-137,19.5,1.55,UNDER,0.005,0.995,"(0.3, 16.2)",OVER/UNDER/UNDER,1,0.8588,4.153,0.831
48498,Jarrett Allen,player_points,Underdog,-137,12.5,3.31,UNDER,0.069,0.931,"(0.3, 14.9)",Lauri Markkanen,player_points,Underdog,-137,21.5,14.34,UNDER,0.081,0.919,"(4.3, 24.4)",Trey Murphy III,player_points,Underdog,-137,19.5,1.55,UNDER,0.005,0.995,"(0.3, 16.2)",UNDER/UNDER/UNDER,1,0.8513,4.108,0.822
47937,James Harden,player_points,Underdog,-137,20.5,28.37,OVER,0.927,0.073,"(18.0, 38.7)",Lauri Markkanen,player_points,Underdog,-137,21.5,14.34,UNDER,0.081,0.919,"(4.3, 24.4)",Trey Murphy III,player_points,Underdog,-137,19.5,1.55,UNDER,0.005,0.995,"(0.3, 16.2)",OVER/UNDER/UNDER,1,0.8472,4.083,0.817
20351,Bobby Portis,player_points,Underdog,-137,10.5,15.92,OVER,0.886,0.114,"(7.3, 24.5)",Mikal Bridges,player_points,Underdog,-137,15.5,4.02,UNDER,0.044,0.956,"(0.4, 17.2)",Trey Murphy III,player_points,Underdog,-137,19.5,1.55,UNDER,0.005,0.995,"(0.3, 16.2)",OVER/UNDER/UNDER,1,0.8426,4.056,0.811
48330,Jarrett Allen,player_points,Underdog,-137,12.5,3.31,UNDER,0.069,0.931,"(0.3, 14.9)",Julian Champagnie,player_points,Underdog,-137,10.5,16.91,OVER,0.907,0.093,"(7.5, 26.4)",Trey Murphy III,player_points,Underdog,-137,19.5,1.55,UNDER,0.005,0.995,"(0.3, 16.2)",UNDER/OVER/UNDER,1,0.8405,4.043,0.809


In [12]:
threeLeg = prizepicks3LegEV(
    data=dfREB,
    bookmakers=dfsREB,
    model=REBmodel,
    features=REBfeatures,
    edge_threshold=2.0,
    stake=100,
    simulations=10000,
    std_window=10,
    min_std=1.5,
    max_std=6.5,
    stat_col='REB'
)
threeLeg.sort_values(by='EV%', ascending=False).head(10)

Processing 3-leg parlays...


,PLAYER 1,CATEGORY 1,BOOKMAKER 1,ODDS 1,LINE 1,PREDICTION 1,MODEL_SIDE 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,BOOKMAKER 2,ODDS 2,LINE 2,PREDICTION 2,MODEL_SIDE 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,PLAYER 3,CATEGORY 3,BOOKMAKER 3,ODDS 3,LINE 3,PREDICTION 3,MODEL_SIDE 3,OVER% 3,UNDER% 3,CONFIDENCE INTERVAL 3,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
7201,Josh Hart,player_rebounds,prizepicks,-137,9.5,7.59,UNDER,0.243,0.757,"(2.4, 12.9)",Precious Achiuwa,player_rebounds,prizepicks,-137,8.5,7.05,UNDER,0.234,0.766,"(3.1, 10.9)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",UNDER/UNDER/OVER,0,0.4678,1.807,0.361
8216,Pascal Siakam,player_rebounds,prizepicks,-137,7.5,5.85,UNDER,0.256,0.744,"(1.3, 10.7)",Precious Achiuwa,player_rebounds,prizepicks,-137,8.5,7.05,UNDER,0.234,0.766,"(3.1, 10.9)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",UNDER/UNDER/OVER,0,0.4596,1.757,0.351
8315,Precious Achiuwa,player_rebounds,prizepicks,-137,8.5,7.05,UNDER,0.234,0.766,"(3.1, 10.9)",Thomas Bryant,player_rebounds,prizepicks,-137,6.5,5.14,UNDER,0.259,0.741,"(1.3, 9.1)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",UNDER/UNDER/OVER,0,0.4576,1.746,0.349
6055,Jalen Duren,player_rebounds,prizepicks,-137,11.5,14.22,OVER,0.739,0.261,"(5.7, 22.8)",Precious Achiuwa,player_rebounds,prizepicks,-137,8.5,7.05,UNDER,0.234,0.766,"(3.1, 10.9)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",OVER/UNDER/OVER,0,0.4568,1.741,0.348
7177,Josh Hart,player_rebounds,prizepicks,-137,9.5,7.59,UNDER,0.243,0.757,"(2.4, 12.9)",Pascal Siakam,player_rebounds,prizepicks,-137,7.5,5.85,UNDER,0.256,0.744,"(1.3, 10.7)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",UNDER/UNDER/OVER,0,0.4543,1.726,0.345
5084,Immanuel Quickley,player_rebounds,prizepicks,-137,3.5,4.89,OVER,0.735,0.265,"(0.8, 9.6)",Precious Achiuwa,player_rebounds,prizepicks,-137,8.5,7.05,UNDER,0.234,0.766,"(3.1, 10.9)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",OVER/UNDER/OVER,0,0.4541,1.724,0.345
8306,Precious Achiuwa,player_rebounds,prizepicks,-137,8.5,7.05,UNDER,0.234,0.766,"(3.1, 10.9)",Santi Aldama,player_rebounds,prizepicks,-137,5.5,7.50,OVER,0.733,0.267,"(1.6, 14.3)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",UNDER/OVER/OVER,0,0.4531,1.719,0.344
7216,Josh Hart,player_rebounds,prizepicks,-137,9.5,7.59,UNDER,0.243,0.757,"(2.4, 12.9)",Thomas Bryant,player_rebounds,prizepicks,-137,6.5,5.14,UNDER,0.259,0.741,"(1.3, 9.1)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",UNDER/UNDER/OVER,0,0.4524,1.714,0.343
5887,Jalen Duren,player_rebounds,prizepicks,-137,11.5,14.22,OVER,0.739,0.261,"(5.7, 22.8)",Josh Hart,player_rebounds,prizepicks,-137,9.5,7.59,UNDER,0.243,0.757,"(2.4, 12.9)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",OVER/UNDER/OVER,0,0.4516,1.710,0.342
4916,Immanuel Quickley,player_rebounds,prizepicks,-137,3.5,4.89,OVER,0.735,0.265,"(0.8, 9.6)",Josh Hart,player_rebounds,prizepicks,-137,9.5,7.59,UNDER,0.243,0.757,"(2.4, 12.9)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",OVER/UNDER/OVER,0,0.4488,1.693,0.339


In [43]:
from nba_api.stats.endpoints import playbyplayv3
import isodate

dfv3 = playbyplayv3.PlayByPlayV3(game_id='0022300068').get_data_frames()[0]
dfv3['MIN_SECONDS'] = dfv3['clock'].apply(lambda x: isodate.parse_duration(x).total_seconds())
dfv3['scoreHome'] = dfv3['scoreHome'].ffill()
dfv3['scoreAway'] = dfv3['scoreAway'].ffill()
dfv3


,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId,MIN_SECONDS
0,0022300068,2,PT12M00.00S,1,0,,0,,,0,0,0,,0,0,0,0,,Start of 1st Period (7:45 PM EST),period,start,0,0,1,720.0
1,0022300068,4,PT12M00.00S,1,1610612748,MIA,1628389,Adebayo,B. Adebayo,0,0,0,,0,,,0,h,Jump Ball Adebayo vs. Duren: Tip to Thompson,Jump Ball,,1,0,2,720.0
2,0022300068,7,PT11M43.00S,1,1610612765,DET,1630191,Stewart,I. Stewart,-6,45,5,Made,1,0,2,2,v,Stewart 5' Turnaround Hook Shot (2 PTS) (Duren...,Made Shot,Turnaround Hook Shot,1,2,3,703.0
3,0022300068,9,PT11M20.00S,1,1610612748,MIA,1629639,Herro,T. Herro,-116,120,17,Missed,1,,,0,h,MISS Herro 17' Pullup Jump Shot,Missed Shot,Pullup Jump shot,1,2,4,680.0
4,0022300068,10,PT11M18.00S,1,1610612765,DET,1631105,Duren,J. Duren,0,0,0,,0,,,0,v,Duren REBOUND (Off:0 Def:1),Rebound,Unknown,1,0,5,678.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
509,0022300068,696,PT00M02.50S,4,1610612748,MIA,201567,Love,K. Love,0,0,0,,0,,,0,h,SUB: Martin FOR Love,Substitution,,0,0,510,2.5
510,0022300068,697,PT00M02.50S,4,1610612765,DET,1630191,Stewart,I. Stewart,0,0,0,,0,,,0,v,SUB: Harris FOR Stewart,Substitution,,0,0,511,2.5
511,0022300068,700,PT00M00.60S,4,1610612765,DET,1630595,Cunningham,C. Cunningham,-56,298,30,Missed,1,,,0,v,MISS Cunningham 30' 3PT Jump Shot,Missed Shot,Jump Shot,1,3,512,0.6
512,0022300068,701,PT00M00.10S,4,0,,1610612765,,,0,0,0,,0,,,0,v,Pistons Rebound,Rebound,Unknown,1,0,513,0.1
